# Chapter 8.5 - Batch Normalization

Batch normalization adds a trainable normalization layer inside a network. It normalizes activations during training, learns a scale and shift, and stores running statistics for evaluation. The concept is simple; the framework behavior is stateful and worth inspecting carefully.

## How to use this notebook

Run the notebook from top to bottom in a clean kernel. The code uses small synthetic tensors so that architecture mechanics can be inspected without downloads, `torchvision`, ImageNet-scale images, or long training runs. Before important cells, predict the shape, parameter count, or failure mode, then read the assertions as executable contracts.

## You are done when you can

- compute batch normalization manually for dense activations
- explain which axes are normalized in convolutional feature maps
- distinguish training behavior from evaluation behavior
- identify gamma, beta, running mean, and running variance in PyTorch
- debug batch normalization when a training batch has too little data per feature


In [ ]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_parameters(module):
    return sum(p.numel() for p in module.parameters())

def trace_module_shapes(module, X):
    rows = []
    current = X
    for name, layer in module.named_children():
        current = layer(current)
        rows.append((name, layer.__class__.__name__, shape(current)))
    return rows, current


## 8.5.0 The Problem This Notebook Solves

Deep networks can be hard to train partly because activation distributions can shift as earlier layers update. Batch normalization gives the network a normalization step inside the model:

```text
center activations using batch mean
scale by batch standard deviation
learn gamma and beta to restore useful scale and offset
store running statistics for evaluation
```

The key warning is that batch normalization is not just a formula. It has trainable parameters and persistent buffers. That connects directly to Chapter 6 parameter and buffer management.


## 8.5.1 Manual Batch Normalization for Dense Activations

For a 2D tensor shaped `(batch, features)`, batch normalization computes one mean and variance per feature across the batch.

Then it applies:

```text
X_hat = (X - mean) / sqrt(variance + epsilon)
Y = gamma * X_hat + beta
```

`epsilon` is a small positive number that prevents division by zero.


In [ ]:
def manual_batch_norm_2d(X, gamma, beta, eps=1e-5):
    mean = X.mean(dim=0, keepdim=True)
    var = ((X - mean) ** 2).mean(dim=0, keepdim=True)
    X_hat = (X - mean) / torch.sqrt(var + eps)
    return gamma * X_hat + beta, mean, var


X = torch.tensor([[1.0, 10.0, 100.0],
                  [2.0, 20.0, 200.0],
                  [3.0, 30.0, 300.0]])
gamma = torch.ones(1, 3)
beta = torch.zeros(1, 3)
Y, mean, var = manual_batch_norm_2d(X, gamma, beta)

print("mean:", mean)
print("var:", var)
print("normalized feature means:", Y.mean(dim=0))

assert torch.allclose(Y.mean(dim=0), torch.zeros(3), atol=1e-5)


## 8.5.2 Convolutional BatchNorm Normalizes Per Channel

For image-like tensors shaped `(batch, channels, height, width)`, `BatchNorm2d` computes one mean and variance per channel. It averages over:

```text
batch dimension
height dimension
width dimension
```

It does not compute separate statistics for every pixel location. The same channel scale and shift are shared across spatial locations.


In [ ]:
X = torch.randn(4, 3, 5, 5) * torch.tensor([1.0, 5.0, 10.0]).view(1, 3, 1, 1)
mean = X.mean(dim=(0, 2, 3), keepdim=True)
var = ((X - mean) ** 2).mean(dim=(0, 2, 3), keepdim=True)
X_hat = (X - mean) / torch.sqrt(var + 1e-5)

channel_means = X_hat.mean(dim=(0, 2, 3))
channel_vars = X_hat.var(dim=(0, 2, 3), unbiased=False)

print("channel means:", channel_means)
print("channel vars:", channel_vars)

assert torch.allclose(channel_means, torch.zeros(3), atol=1e-5)
assert torch.allclose(channel_vars, torch.ones(3), atol=1e-4)


## 8.5.3 PyTorch BatchNorm Has Parameters and Buffers

`nn.BatchNorm1d(3)` owns:

- `weight`: learnable gamma, initialized to 1
- `bias`: learnable beta, initialized to 0
- `running_mean`: buffer, not usually learned by gradient descent
- `running_var`: buffer, not usually learned by gradient descent

Parameters are updated by the optimizer. Buffers are saved and moved with the module, but they are not optimizer parameters.


In [ ]:
bn = nn.BatchNorm1d(3)

print("parameters:", list(dict(bn.named_parameters()).keys()))
print("buffers:", list(dict(bn.named_buffers()).keys()))
print("state_dict:", list(bn.state_dict().keys()))

assert set(dict(bn.named_parameters())) == {"weight", "bias"}
assert {"running_mean", "running_var"}.issubset(dict(bn.named_buffers()))
assert "running_mean" in bn.state_dict()


## 8.5.4 Training Mode Updates Running Statistics; Eval Mode Uses Them

BatchNorm behaves differently in training and evaluation:

- In training mode, it normalizes using the current batch and updates running statistics.
- In evaluation mode, it normalizes using stored running statistics.

This is why `model.train()` and `model.eval()` are not optional ceremony. They change model behavior.


In [ ]:
bn = nn.BatchNorm1d(3, momentum=0.5)
before = bn.running_mean.clone()

bn.train()
train_batch = torch.tensor([[10.0, 20.0, 30.0],
                            [12.0, 22.0, 32.0],
                            [14.0, 24.0, 34.0]])
train_output = bn(train_batch)
after = bn.running_mean.clone()

bn.eval()
eval_batch = torch.tensor([[100.0, 200.0, 300.0],
                           [101.0, 201.0, 301.0]])
eval_output = bn(eval_batch)

print("running mean before:", before)
print("running mean after:", after)
print("eval output mean:", eval_output.mean(dim=0))

assert not torch.equal(before, after)
assert not torch.allclose(eval_output.mean(dim=0), torch.zeros(3), atol=1e-2)


## 8.5.5 BatchNorm Inside a CNN Block

In CNNs, batch normalization usually appears after convolution and before activation:

```text
Conv2d -> BatchNorm2d -> ReLU
```

The convolution creates feature maps. BatchNorm stabilizes channel scale. ReLU adds nonlinearity. The batch norm layer has trainable gamma/beta and running-stat buffers, so it participates in both optimization and checkpoint state.


In [ ]:
conv_bn_relu = nn.Sequential(
    nn.Conv2d(1, 4, kernel_size=3, padding=1, bias=False),
    nn.BatchNorm2d(4),
    nn.ReLU(),
)

X = torch.randn(2, 1, 8, 8)
Y = conv_bn_relu(X)
loss = Y.pow(2).mean()
loss.backward()

state_keys = list(conv_bn_relu.state_dict().keys())
print("output shape:", shape(Y))
print("state keys:", state_keys)

assert shape(Y) == (2, 4, 8, 8)
assert conv_bn_relu[1].weight.grad is not None
assert "1.running_mean" in state_keys


## 8.5.6 Break It Deliberately: Too Little Data Per Feature

During training, BatchNorm needs enough values to estimate a variance for each normalized feature. For `BatchNorm1d` with input shape `(batch, features)`, a batch size of 1 gives only one value per feature.

PyTorch rejects that in training mode because the batch statistics would be degenerate.


In [ ]:
bn = nn.BatchNorm1d(3)
bn.train()

try:
    bn(torch.randn(1, 3))
except ValueError as error:
    print(type(error).__name__)
    print(str(error).splitlines()[0])
else:
    raise AssertionError("Expected BatchNorm1d to reject one value per feature in training")


## 8.5 Checkpoint

Answer these before moving on. Short markdown answers in the notebook are enough; the checkpoint is meant to test whether you can explain the mechanics without rereading the code.

1. What does batch normalization normalize in a 2D dense activation tensor?
2. Which axes does `BatchNorm2d` reduce over for image-like feature maps?
3. What are gamma and beta?
4. Why are running mean and running variance buffers rather than optimizer parameters?
5. Why can forgetting `model.eval()` change inference behavior?
